# Refit the stacked logical-error comparison

Read the Cornucopia, bivariate-bicycle, and surface-code decode summaries,
export seven X/Z-averaged data and fit tables, and generate only
`stacked_error_rate_fits.pdf`. Panel a shows Cornucopia codes; panel b compares
Cornucopia, BB, and surface codes. The shared plotter uses DejaVu Sans.

The three raw summaries are required but are not supplied. Use
`plot_error_rates.ipynb` to reproduce the same figure from the supplied CSVs.

For block failure probability $P_B$, $k$ logical qubits, and $T$ cycles,
the effective rate is $p_L=1-(1-P_B)^{1/(kT)}$. X/Z averaging takes place
at the block-rate level before this conversion; it does not establish
independence of logical-qubit errors. Error bars use the counting approximation
$p_L/\sqrt{N_{\mathrm{fail},X}+N_{\mathrm{fail},Z}}$.

All families use `xz` detector mode. Fits are unweighted least squares in log
space, $\log p_L=(d/2)\log p+c_0+c_1p+c_2p^2$. Cornucopia uses
$p\leq0.0025$; BB and surface use $p\leq0.005$. These finite-distance fits
are not threshold estimates, and extrapolation does not constitute direct
measurement of the predicted error rates.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "code_construction" / "affine_codes.py").is_file()
)
sys.path.insert(0, str(REPO_ROOT))

In [ ]:
import csv
import math
import re
from collections import defaultdict

import numpy as np

ROOT = REPO_ROOT / "circuit_simulation" / "results"
SUMMARY_TXT = ROOT / "decode" / "summary.txt"
OUT_DIR = REPO_ROOT / "figures" / "logical_error_rates" / "results"
required_summaries = [
    SUMMARY_TXT,
    REPO_ROOT / "bivariate_bicycle" / "results" / "decode" / "summary.txt",
    REPO_ROOT / "surface_code" / "results" / "decode" / "summary.txt",
]
missing = [
    path.relative_to(REPO_ROOT).as_posix()
    for path in required_summaries
    if not path.is_file()
]
if missing:
    raise FileNotFoundError(
        "Raw decode summaries are required: "
        + ", ".join(missing)
        + ". Use plot_error_rates.ipynb to plot the supplied CSV tables."
    )
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODE = "xz"
Y_COLUMN = "LER_per_cycle_per_logical"
MIN_SHOTS = 1
EXCLUDE_P_VALUES = {0.0045}
EXCLUDE_EXPECTED_D_VALUES = set()

print(f"summary_txt={SUMMARY_TXT}")
print(f"output_dir={OUT_DIR}")

## Summary parsing and shared numerical helpers

In [ ]:
def _as_float(value):
    if value is None:
        return None
    if isinstance(value, str) and value.strip() == "None":
        return None
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def _as_int(value):
    if value is None:
        return None
    if isinstance(value, str) and value.strip() == "None":
        return None
    try:
        return int(value)
    except (TypeError, ValueError):
        return None


def _parse_scalar(value: str):
    value = value.strip()
    if value == "None":
        return None
    if re.fullmatch(r"[+-]?\d+", value):
        return int(value)
    try:
        return float(value)
    except ValueError:
        return value


FLOAT_KEYS = [
    "LER",
    "LER_per_cycle",
    "LER_per_cycle_per_logical",
    "avg_iterations",
    "c_LER",
    "c_LER_per_cycle_per_logical",
    "c_avg_iterations",
    "c_direct_per_logical_LER",
    "convergence_rate",
    "direct_per_logical_LER",
    "u_LER",
    "u_LER_per_cycle_per_logical",
    "u_avg_iterations",
    "u_direct_per_logical_LER",
]
INT_KEYS = [
    "c_failures",
    "c_max_iterations",
    "c_observable_mismatches",
    "converged",
    "failures",
    "max_iterations",
    "observable_mismatches",
    "shots",
    "u_failures",
    "u_max_iterations",
    "u_observable_mismatches",
    "unconverged",
]


def _blank_record(
    mode: str | None, code_info: dict[str, object], label: str
) -> dict[str, object]:
    return {
        "mode": mode,
        "code_name": code_info.get("code_name"),
        "parameter_label": code_info.get("parameter_label"),
        "P": code_info.get("P"),
        "expected_d": code_info.get("expected_d"),
        "basis": None,
        "p": None,
        "cycles": None,
        "shots": None,
        "failures": None,
        "LER": None,
        "LER_per_cycle": None,
        "LER_per_cycle_per_logical": None,
        "direct_per_logical_LER": None,
        "converged": None,
        "unconverged": None,
        "convergence_rate": None,
        "c_failures": None,
        "c_LER": None,
        "c_LER_per_cycle_per_logical": None,
        "c_observable_mismatches": None,
        "c_direct_per_logical_LER": None,
        "u_failures": None,
        "u_LER": None,
        "u_LER_per_cycle_per_logical": None,
        "u_observable_mismatches": None,
        "u_direct_per_logical_LER": None,
        "detectors": None,
        "observables": None,
        "label": label,
    }


def load_summary_txt(path: Path) -> list[dict[str, object]]:
    """Parse decode/summary.txt into the same record shape used by plots."""
    records: list[dict[str, object]] = []
    current_mode: str | None = None
    current_code: dict[str, object] = {}
    current: dict[str, object] | None = None
    section: str | None = None

    code_re = re.compile(
        r"^code=(?P<code>\S+)\s+(?P<label>\[\[[^\]]+\]\])\s+P=(?P<P>\d+)\s+expected_d=(?P<d>\d+)\s*$"
    )
    condition_re = re.compile(r"^(?P<label>cornucopia_\S+_[zx]basis_\S+)$")
    meta_re = re.compile(
        r"^meta:\s+basis=(?P<basis>\S+)\s+p=(?P<p>\S+)\s+cycles=(?P<cycles>\d+)\s+"
        r"detectors=(?P<detectors>\d+)\s+observables=(?P<observables>\d+)\s+"
        r"convergence_rate=(?P<convergence_rate>\S+)"
    )

    def finish_current() -> None:
        nonlocal current
        if current is not None:
            records.append(current)
            current = None

    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.rstrip()
        stripped = line.strip()
        if not stripped:
            continue

        if stripped.startswith("## mode="):
            current_mode = stripped.split("=", 1)[1].strip()
            continue

        m = code_re.match(stripped)
        if m:
            finish_current()
            current_code = {
                "code_name": m.group("code"),
                "parameter_label": m.group("label"),
                "P": int(m.group("P")),
                "expected_d": int(m.group("d")),
            }
            section = None
            continue

        m = condition_re.match(stripped)
        if m and current_code:
            finish_current()
            current = _blank_record(current_mode, current_code, m.group("label"))
            section = None
            continue

        if current is None:
            continue

        m = meta_re.match(stripped)
        if m:
            current.update(
                {
                    "basis": m.group("basis"),
                    "p": _as_float(m.group("p")),
                    "cycles": _as_int(m.group("cycles")),
                    "detectors": _as_int(m.group("detectors")),
                    "observables": _as_int(m.group("observables")),
                    "convergence_rate": _as_float(m.group("convergence_rate")),
                }
            )
            section = None
            continue

        if not line.startswith(" ") and stripped.endswith(":"):
            section = stripped[:-1]
            continue

        if not line.startswith("  ") or section is None:
            continue

        parts = stripped.split()
        if len(parts) < 2:
            continue
        key = parts[0]
        value = _parse_scalar(" ".join(parts[1:]))

        if section == "all shots":
            mapping = {
                "shots": "shots",
                "failures": "failures",
                "LER": "LER",
                "LER_per_cycle": "LER_per_cycle",
                "LER_per_cycle_per_logical": "LER_per_cycle_per_logical",
                "observable_mismatches": "observable_mismatches",
                "direct_per_logical_LER": "direct_per_logical_LER",
                "avg_iterations": "avg_iterations",
                "max_iterations": "max_iterations",
            }
        elif section == "relaybp-converged after fallback shots":
            mapping = {
                "converged": "converged",
                "c_failures": "c_failures",
                "c_LER": "c_LER",
                "c_LER_per_cycle_per_logical": "c_LER_per_cycle_per_logical",
                "c_observable_mismatches": "c_observable_mismatches",
                "c_direct_per_logical_LER": "c_direct_per_logical_LER",
                "c_avg_iterations": "c_avg_iterations",
                "c_max_iterations": "c_max_iterations",
            }
        elif section == "still relaybp-unconverged before MIP shots":
            mapping = {
                "unconverged": "unconverged",
                "u_failures": "u_failures",
                "u_LER": "u_LER",
                "u_LER_per_cycle_per_logical": "u_LER_per_cycle_per_logical",
                "u_observable_mismatches": "u_observable_mismatches",
                "u_direct_per_logical_LER": "u_direct_per_logical_LER",
                "u_avg_iterations": "u_avg_iterations",
                "u_max_iterations": "u_max_iterations",
            }
        else:
            mapping = {}

        out_key = mapping.get(key)
        if out_key:
            if out_key in FLOAT_KEYS:
                current[out_key] = _as_float(value)
            elif out_key in INT_KEYS:
                current[out_key] = _as_int(value)
            else:
                current[out_key] = value

    finish_current()
    return records


records = load_summary_txt(SUMMARY_TXT)

In [ ]:
def grouped_by_code(
    records: list[dict[str, object]],
) -> dict[tuple[int, int, str, str], list[dict[str, object]]]:
    groups: dict[tuple[int, int, str, str], list[dict[str, object]]] = defaultdict(list)
    for r in records:
        key = (
            int(r["expected_d"]),
            int(r["P"]),
            str(r["code_name"]),
            str(r["parameter_label"]),
        )
        groups[key].append(r)
    return {
        key: sorted(value, key=lambda r: r["p"])
        for key, value in sorted(groups.items())
    }


COMMON_AXIS_CURVE_MIN_P = 0.001


def failure_column_for_y(y_column: str) -> str:
    if y_column.startswith("c_"):
        return "c_failures"
    if y_column.startswith("u_"):
        return "u_failures"
    return "failures"


def yerr_from_failures(record: dict[str, object], y_column: str) -> float:
    y_value = record.get(y_column)
    failures = record.get(failure_column_for_y(y_column))
    if not (isinstance(y_value, float) and y_value > 0):
        return 0.0
    if not (isinstance(failures, int) and failures > 0):
        return 0.0
    return float(y_value) / math.sqrt(failures)

## Distance model and fit-table exports

In [ ]:
FIT_EXPECTED_D_TO_FIT_D = {
    6: 6,
    8: 8,
    10: 10,
    12: 12,
    14: 14,
    16: 16,
    18: 18,
}
FIT_EXTRA_CODE_NAMES = {"cornucopia_p237_d18"}
FIT_Y_COLUMN = Y_COLUMN
FIT_WEIGHT_MODE = "unweighted_log"  # per-code log-space least squares; alternative: failure_weighted_log
FIT_MAX_P_FOR_FIT = 0.0025
FIT_CURVE_MIN_P = COMMON_AXIS_CURVE_MIN_P


def include_in_fitted_distance_model(record: dict[str, object]) -> bool:
    expected_d = record.get("expected_d")
    code_name = record.get("code_name")
    if expected_d in {6, 8, 10, 12, 14, 16}:
        return True
    return code_name in FIT_EXTRA_CODE_NAMES


def fit_model_rows(records: list[dict[str, object]]) -> list[dict[str, object]]:
    rows = []
    for r in records:
        expected_d = r.get("expected_d")
        p_value = r.get("p")
        y_value = r.get(FIT_Y_COLUMN)
        failures = r.get("failures")
        if (
            expected_d not in FIT_EXPECTED_D_TO_FIT_D
            or not include_in_fitted_distance_model(r)
        ):
            continue
        if not (isinstance(p_value, float) and p_value > 0):
            continue
        if not (isinstance(y_value, float) and y_value > 0):
            continue
        if not (isinstance(failures, int) and failures > 0):
            continue
        if p_value > FIT_MAX_P_FOR_FIT:
            continue
        rows.append(r)
    return sorted(rows, key=lambda r: (r["expected_d"], r["P"], r["p"]))


def plot_model_rows(records: list[dict[str, object]]) -> list[dict[str, object]]:
    rows = []
    for r in records:
        expected_d = r.get("expected_d")
        p_value = r.get("p")
        y_value = r.get(FIT_Y_COLUMN)
        failures = r.get("failures")
        if (
            expected_d not in FIT_EXPECTED_D_TO_FIT_D
            or not include_in_fitted_distance_model(r)
        ):
            continue
        if not (isinstance(p_value, float) and p_value > 0):
            continue
        if not (isinstance(y_value, float) and y_value > 0):
            continue
        if not (isinstance(failures, int) and failures > 0):
            continue
        rows.append(r)
    return sorted(rows, key=lambda r: (r["expected_d"], r["P"], r["p"]))


def distance_fit(
    rows: list[dict[str, object]], fit_d: int
) -> tuple[np.ndarray, dict[str, float]]:
    if len(rows) < 2:
        raise ValueError(f"Need at least 2 data points for fit, got {len(rows)}")
    p = np.asarray([float(r["p"]) for r in rows], dtype=float)
    y = np.asarray([float(r[FIT_Y_COLUMN]) for r in rows], dtype=float)
    failures = np.asarray([max(int(r["failures"]), 1) for r in rows], dtype=float)
    target = np.log(y) - 0.5 * float(fit_d) * np.log(p)
    fit_degree = min(2, len(rows) - 1)
    if fit_degree == 2:
        design = np.column_stack([np.ones_like(p), p, p * p])
    else:
        design = np.column_stack([np.ones_like(p), p])
    if FIT_WEIGHT_MODE == "unweighted_log":
        sqrt_w = np.ones_like(p)
    elif FIT_WEIGHT_MODE == "failure_weighted_log":
        sqrt_w = np.sqrt(failures)
    else:
        raise ValueError(f"unknown FIT_WEIGHT_MODE={FIT_WEIGHT_MODE!r}")
    coeff_raw, *_ = np.linalg.lstsq(
        design * sqrt_w[:, None], target * sqrt_w, rcond=None
    )
    pred_target = design @ coeff_raw
    if fit_degree == 2:
        coeff = coeff_raw
    else:
        coeff = np.asarray([float(coeff_raw[0]), float(coeff_raw[1]), 0.0], dtype=float)
    pred_log_y = 0.5 * float(fit_d) * np.log(p) + pred_target
    residual = np.log(y) - pred_log_y
    if FIT_WEIGHT_MODE == "failure_weighted_log":
        rmse_log = float(np.sqrt(np.average(residual * residual, weights=failures)))
    else:
        rmse_log = float(np.sqrt(np.mean(residual * residual)))
    stats = {
        "weight_mode": FIT_WEIGHT_MODE,
        "rmse_log": rmse_log,
        "max_abs_residual_log": float(np.max(np.abs(residual))),
        "num_points": int(len(rows)),
        "fit_degree": int(fit_degree),
        "failure_weight_sum": int(np.sum(failures)),
    }
    return coeff, stats


def model_curve(p: np.ndarray, fit_d: int, coeff: np.ndarray) -> np.ndarray:
    c0, c1, c2 = [float(x) for x in coeff]
    return np.exp(0.5 * float(fit_d) * np.log(p) + c0 + c1 * p + c2 * p * p)


def export_distance_fit(records, save_prefix: str) -> None:
    fit_rows = fit_model_rows(records)
    plot_rows = plot_model_rows(records)
    groups = grouped_by_code(fit_rows)
    plot_groups = grouped_by_code(plot_rows)
    groups = {
        key: value for key, value in groups.items() if key[0] in FIT_EXPECTED_D_TO_FIT_D
    }
    if not groups:
        raise RuntimeError("No records available for distance model fitting")
    coeff_rows = [
        [
            "expected_d",
            "fit_d",
            "P",
            "code_name",
            "parameter_label",
            "weight_mode",
            "fit_max_p",
            "curve_min_p",
            "num_points",
            "fit_degree",
            "failure_weight_sum",
            "c0",
            "c1",
            "c2",
            "rmse_log",
            "max_abs_residual_log",
        ]
    ]
    data_rows = [
        [
            "expected_d",
            "fit_d",
            "P",
            "code_name",
            "parameter_label",
            "fit_max_p",
            "used_for_fit",
            "p",
            "y",
            "yerr",
            "failures",
            "shots",
            "fit_y",
            "log_residual",
        ]
    ]
    for (expected_d, P, code_name, parameter_label), rows in groups.items():
        fit_d = FIT_EXPECTED_D_TO_FIT_D[int(expected_d)]
        coeff, stats = distance_fit(rows, fit_d)
        plot_rows_for_group = plot_groups.get(
            (expected_d, P, code_name, parameter_label), rows
        )
        xs = np.asarray([float(r["p"]) for r in plot_rows_for_group], dtype=float)
        ys = np.asarray(
            [float(r[FIT_Y_COLUMN]) for r in plot_rows_for_group], dtype=float
        )
        yerrs = np.asarray(
            [yerr_from_failures(r, FIT_Y_COLUMN) for r in plot_rows_for_group],
            dtype=float,
        )
        fit_at_x = model_curve(xs, fit_d, coeff)
        residuals = np.log(ys) - np.log(fit_at_x)
        coeff_rows.append(
            [
                str(expected_d),
                str(fit_d),
                str(P),
                str(code_name),
                str(parameter_label),
                str(stats["weight_mode"]),
                f"{FIT_MAX_P_FOR_FIT:.16g}",
                f"{FIT_CURVE_MIN_P:.16g}",
                str(stats["num_points"]),
                str(stats["fit_degree"]),
                str(stats["failure_weight_sum"]),
                f"{coeff[0]:.16g}",
                f"{coeff[1]:.16g}",
                f"{coeff[2]:.16g}",
                f"{stats['rmse_log']:.16g}",
                f"{stats['max_abs_residual_log']:.16g}",
            ]
        )
        for r, y, yerr, fy, residual in zip(
            plot_rows_for_group, ys, yerrs, fit_at_x, residuals, strict=True
        ):
            p_for_row = float(r["p"])
            used_for_fit = p_for_row <= FIT_MAX_P_FOR_FIT
            data_rows.append(
                [
                    str(expected_d),
                    str(fit_d),
                    str(P),
                    str(code_name),
                    str(parameter_label),
                    f"{FIT_MAX_P_FOR_FIT:.16g}",
                    str(used_for_fit),
                    f"{p_for_row:.16g}",
                    f"{y:.16g}",
                    f"{yerr:.16g}",
                    str(r["failures"]),
                    str(r["shots"]),
                    f"{fy:.16g}",
                    f"{residual:.16g}",
                ]
            )
        print(
            f"fit {parameter_label}: weight_mode={stats['weight_mode']} fit_p<= {FIT_MAX_P_FOR_FIT:g} d={fit_d} "
            f"points={stats['num_points']} degree={stats['fit_degree']} c0={coeff[0]:.6g} "
            f"c1={coeff[1]:.6g} c2={coeff[2]:.6g} rmse_log={stats['rmse_log']:.4g}"
        )
    coeff_path = OUT_DIR / f"{save_prefix}_coefficients.csv"
    data_path = OUT_DIR / f"{save_prefix}_data.csv"
    for path, rows_to_write in [(coeff_path, coeff_rows), (data_path, data_rows)]:
        with path.open("w", newline="") as f:
            csv.writer(f).writerows(rows_to_write)
        print(f"wrote {path}")

## Cornucopia X/Z average and fit exports

In [ ]:
AVERAGE_SAVE_PREFIX = "cornucopia_xz_average_fit"
AVERAGE_POINTS_PATH = OUT_DIR / "cornucopia_xz_average_rates.csv"


def _valid_average_source(
    record: dict[str, object],
    excluded_p_values: set[float],
) -> bool:
    p_value = record.get("p")
    ler = record.get("LER")
    shots = record.get("shots")
    failures = record.get("failures")
    cycles = record.get("cycles")
    observables = record.get("observables")
    return (
        record.get("mode") == MODE
        and record.get("basis") in {"X", "Z"}
        and isinstance(p_value, float)
        and p_value > 0
        and p_value not in excluded_p_values
        and record.get("expected_d") not in EXCLUDE_EXPECTED_D_VALUES
        and isinstance(ler, float)
        and 0.0 <= ler <= 1.0
        and isinstance(shots, int)
        and shots >= MIN_SHOTS
        and isinstance(failures, int)
        and failures >= 0
        and isinstance(cycles, int)
        and cycles > 0
        and isinstance(observables, int)
        and observables > 0
    )


def build_xz_average_records(
    source_records: list[dict[str, object]],
    excluded_p_values: set[float] | None = None,
) -> tuple[list[dict[str, object]], list[tuple[object, ...]]]:
    if excluded_p_values is None:
        excluded_p_values = EXCLUDE_P_VALUES
    pairs: dict[tuple[object, ...], dict[str, dict[str, object]]] = {}
    for record in source_records:
        if not _valid_average_source(record, excluded_p_values):
            continue
        key = (
            record.get("code_name"),
            record.get("parameter_label"),
            record.get("P"),
            record.get("expected_d"),
            record.get("p"),
            record.get("cycles"),
        )
        basis = str(record["basis"])
        pair = pairs.setdefault(key, {})
        if basis in pair:
            raise RuntimeError(
                f"Duplicate {basis}-basis record for average key={key!r}"
            )
        pair[basis] = record

    averaged: list[dict[str, object]] = []
    incomplete: list[tuple[object, ...]] = []
    for key, pair in sorted(
        pairs.items(),
        key=lambda item: (
            item[0][3],
            item[0][2],
            item[0][4],
            item[0][5],
        ),
    ):
        if set(pair) != {"X", "Z"}:
            incomplete.append(key)
            continue

        x_record = pair["X"]
        z_record = pair["Z"]
        x_observables = int(x_record["observables"])
        z_observables = int(z_record["observables"])
        if x_observables != z_observables:
            raise RuntimeError(
                f"X/Z observable-count mismatch for key={key!r}: "
                f"X={x_observables}, Z={z_observables}"
            )

        cycles = int(x_record["cycles"])
        x_ler = float(x_record["LER"])
        z_ler = float(z_record["LER"])
        average_ler = 0.5 * (x_ler + z_ler)
        if average_ler >= 1.0:
            per_cycle = 1.0
            per_logical_per_cycle = 1.0
        else:
            per_cycle = -math.expm1(math.log1p(-average_ler) / cycles)
            per_logical_per_cycle = -math.expm1(
                math.log1p(-average_ler) / (cycles * x_observables)
            )

        x_failures = int(x_record["failures"])
        z_failures = int(z_record["failures"])
        total_failures = x_failures + z_failures
        x_shots = int(x_record["shots"])
        z_shots = int(z_record["shots"])

        average_record = dict(z_record)
        average_record.update(
            {
                "basis": "XZ-average",
                "label": (
                    f"{x_record['code_name']}_xzbasis_average_"
                    f"p{float(x_record['p']):.16g}_c{cycles}"
                ),
                "shots": x_shots + z_shots,
                "failures": total_failures,
                "LER": average_ler,
                "LER_per_cycle": per_cycle,
                "LER_per_cycle_per_logical": per_logical_per_cycle,
                "x_LER": x_ler,
                "z_LER": z_ler,
                "x_shots": x_shots,
                "z_shots": z_shots,
                "x_failures": x_failures,
                "z_failures": z_failures,
                "detectors": None,
            }
        )
        averaged.append(average_record)

    return averaged, incomplete


def write_xz_average_points(
    averaged_records: list[dict[str, object]], path: Path
) -> None:
    columns = [
        "code_name",
        "parameter_label",
        "P",
        "expected_d",
        "p",
        "cycles",
        "observables",
        "x_shots",
        "z_shots",
        "x_failures",
        "z_failures",
        "failures",
        "x_LER",
        "z_LER",
        "LER",
        "LER_per_cycle_per_logical",
        "yerr",
    ]
    with path.open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=columns)
        writer.writeheader()
        for record in averaged_records:
            row = {column: record.get(column) for column in columns}
            row["yerr"] = yerr_from_failures(record, Y_COLUMN)
            writer.writerow(row)


xz_average_records, xz_average_incomplete = build_xz_average_records(records)
print(
    f"X/Z-basis average matched_points={len(xz_average_records)} "
    f"incomplete_points={len(xz_average_incomplete)}"
)
for key in xz_average_incomplete:
    print(f"X/Z-basis average skipped incomplete key={key!r}")
for record in xz_average_records:
    print(
        f"avg {record['parameter_label']:>16} p={record['p']:.5g} "
        f"X_LER={record['x_LER']:.6g} Z_LER={record['z_LER']:.6g} "
        f"mean_LER={record['LER']:.6g} "
        f"y={record[Y_COLUMN]:.6g} total_failures={record['failures']}"
    )

if xz_average_records:
    write_xz_average_points(xz_average_records, AVERAGE_POINTS_PATH)
    print(f"wrote {AVERAGE_POINTS_PATH}")
    export_distance_fit(xz_average_records, AVERAGE_SAVE_PREFIX)
else:
    raise RuntimeError(
        "No matched X/Z-basis records available for averaged fitted model."
    )

## BB X/Z average and fit exports

In [ ]:
BB_SUMMARY_TXT = REPO_ROOT / "bivariate_bicycle" / "results" / "decode" / "summary.txt"
BB_FIT_MAX_P = 0.005
BB_FIT_SAVE_PREFIX = "bb_xz_average_fit"
BB_RAW_DATA_PATH = OUT_DIR / "bb_xz_average_rates.csv"


def load_bb_summary_txt(path: Path) -> list[dict[str, object]]:
    records: list[dict[str, object]] = []
    current_mode: str | None = None
    current_code: dict[str, object] = {}
    current: dict[str, object] | None = None
    section: str | None = None

    code_re = re.compile(
        r"^code=(?P<code>\S+)\s+(?P<label>\[\[[^\]]+\]\])\s+"
        r"block_size=(?P<block_size>\d+)\s+expected_d=(?P<d>\d+)\s*$"
    )
    condition_re = re.compile(r"^(?P<label>bb_\S+_[zx]basis_\S+)$")
    meta_re = re.compile(
        r"^meta:\s+basis=(?P<basis>\S+)\s+p=(?P<p>\S+)\s+cycles=(?P<cycles>\d+)\s+"
        r"detectors=(?P<detectors>\d+)\s+observables=(?P<observables>\d+)\s+"
        r"convergence_rate=(?P<convergence_rate>\S+)"
    )

    def finish_current() -> None:
        nonlocal current
        if current is not None:
            records.append(current)
            current = None

    for raw_line in path.read_text(encoding="utf-8").splitlines():
        stripped = raw_line.strip()
        if not stripped:
            continue
        if stripped.startswith("## mode="):
            current_mode = stripped.split("=", 1)[1].strip()
            continue

        code_match = code_re.match(stripped)
        if code_match:
            finish_current()
            current_code = {
                "code_name": code_match.group("code"),
                "parameter_label": code_match.group("label"),
                "block_size": int(code_match.group("block_size")),
                "expected_d": int(code_match.group("d")),
            }
            section = None
            continue

        condition_match = condition_re.match(stripped)
        if condition_match:
            finish_current()
            current = {
                "mode": current_mode,
                **current_code,
                "label": condition_match.group("label"),
                "basis": None,
                "p": None,
                "cycles": None,
                "detectors": None,
                "observables": None,
                "convergence_rate": None,
                "shots": None,
                "failures": None,
                "LER": None,
                "LER_per_cycle_per_logical": None,
            }
            section = None
            continue

        if current is None:
            continue

        meta_match = meta_re.match(stripped)
        if meta_match:
            current.update(
                {
                    "basis": meta_match.group("basis"),
                    "p": float(meta_match.group("p")),
                    "cycles": int(meta_match.group("cycles")),
                    "detectors": int(meta_match.group("detectors")),
                    "observables": int(meta_match.group("observables")),
                    "convergence_rate": float(meta_match.group("convergence_rate")),
                }
            )
            continue

        if stripped.endswith(":"):
            section = stripped[:-1]
            continue

        if section != "all shots":
            continue
        field_match = re.match(r"^(?P<key>\S+)\s+(?P<value>\S+)\s*$", stripped)
        if not field_match:
            continue
        key = field_match.group("key")
        value = field_match.group("value")
        if key in {"shots", "failures", "observable_mismatches"}:
            current[key] = _as_int(value)
        elif key in {
            "LER",
            "LER_per_cycle_per_logical",
            "direct_per_logical_LER",
            "avg_iterations",
        }:
            current[key] = _as_float(value)

    finish_current()
    return records


def valid_bb_plot_record(record: dict[str, object]) -> bool:
    return (
        record.get("mode") == MODE
        and record.get("basis") == "XZ-average"
        and isinstance(record.get("p"), float)
        and float(record["p"]) > 0
        and isinstance(record.get(FIT_Y_COLUMN), float)
        and float(record[FIT_Y_COLUMN]) > 0
        and isinstance(record.get("shots"), int)
        and int(record["shots"]) >= MIN_SHOTS
        and isinstance(record.get("failures"), int)
        and int(record["failures"]) > 0
        and isinstance(record.get("expected_d"), int)
    )


def group_bb_records(
    source_records: list[dict[str, object]],
) -> dict[tuple[int, int, str, str], list[dict[str, object]]]:
    groups: dict[tuple[int, int, str, str], list[dict[str, object]]] = defaultdict(list)
    for record in source_records:
        key = (
            int(record["expected_d"]),
            int(record["block_size"]),
            str(record["code_name"]),
            str(record["parameter_label"]),
        )
        groups[key].append(record)
    for key, rows in groups.items():
        rows.sort(key=lambda row: float(row["p"]))
        p_values = [float(row["p"]) for row in rows]
        if len(p_values) != len(set(p_values)):
            raise RuntimeError(f"Duplicate BB p values for {key!r}: {p_values!r}")
    return dict(sorted(groups.items()))


bb_source_records = load_bb_summary_txt(BB_SUMMARY_TXT)
bb_records, bb_xz_average_incomplete = build_xz_average_records(
    bb_source_records,
    excluded_p_values=set(),
)
print(
    f"BB X/Z-basis average matched_points={len(bb_records)} "
    f"incomplete_points={len(bb_xz_average_incomplete)}"
)
for key in bb_xz_average_incomplete:
    print(f"BB X/Z-basis average skipped incomplete key={key!r}")
bb_plot_records = sorted(
    [record for record in bb_records if valid_bb_plot_record(record)],
    key=lambda record: (
        int(record["expected_d"]),
        int(record["block_size"]),
        float(record["p"]),
    ),
)
bb_fit_records = [
    record for record in bb_plot_records if float(record["p"]) <= BB_FIT_MAX_P
]
bb_plot_groups = group_bb_records(bb_plot_records)
bb_fit_groups = group_bb_records(bb_fit_records)

print(
    f"BB summary={BB_SUMMARY_TXT} source_records={len(bb_source_records)} "
    f"average_records={len(bb_records)} plot_records={len(bb_plot_records)} "
    f"fit_records={len(bb_fit_records)} "
    f"fit_p<={BB_FIT_MAX_P:g}"
)

bb_raw_columns = [
    "code_name",
    "parameter_label",
    "block_size",
    "expected_d",
    "basis",
    "p",
    "x_shots",
    "z_shots",
    "x_failures",
    "z_failures",
    "x_LER",
    "z_LER",
    "cycles",
    "shots",
    "failures",
    "LER",
    "LER_per_cycle_per_logical",
    "yerr",
]
with BB_RAW_DATA_PATH.open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=bb_raw_columns)
    writer.writeheader()
    for record in bb_plot_records:
        row = {column: record.get(column) for column in bb_raw_columns}
        row["yerr"] = yerr_from_failures(record, FIT_Y_COLUMN)
        writer.writerow(row)
print(f"wrote {BB_RAW_DATA_PATH}")
coefficient_rows = [
    [
        "expected_d",
        "block_size",
        "code_name",
        "parameter_label",
        "weight_mode",
        "fit_max_p",
        "curve_min_p",
        "num_points",
        "fit_degree",
        "failure_weight_sum",
        "c0",
        "c1",
        "c2",
        "rmse_log",
        "max_abs_residual_log",
    ]
]
data_rows = [
    [
        "expected_d",
        "block_size",
        "code_name",
        "parameter_label",
        "fit_max_p",
        "used_for_fit",
        "p",
        "y",
        "yerr",
        "failures",
        "shots",
        "fit_y",
        "log_residual",
    ]
]
fitted_group_count = 0

for index, (key, fit_rows) in enumerate(bb_fit_groups.items()):
    expected_d, block_size, code_name, parameter_label = key
    if len(fit_rows) < 2:
        print(f"BB fit skipped {parameter_label}: only {len(fit_rows)} point(s)")
        continue
    plot_rows = bb_plot_groups[key]
    coeff, stats = distance_fit(fit_rows, expected_d)
    xs = np.asarray([float(record["p"]) for record in plot_rows], dtype=float)
    ys = np.asarray([float(record[FIT_Y_COLUMN]) for record in plot_rows], dtype=float)
    yerrs = np.asarray(
        [yerr_from_failures(record, FIT_Y_COLUMN) for record in plot_rows],
        dtype=float,
    )
    fit_at_x = model_curve(xs, expected_d, coeff)
    residuals = np.log(ys) - np.log(fit_at_x)
    fit_xs = np.asarray([float(record["p"]) for record in fit_rows], dtype=float)
    fitted_group_count += 1

    coefficient_rows.append(
        [
            str(expected_d),
            str(block_size),
            code_name,
            parameter_label,
            str(stats["weight_mode"]),
            f"{BB_FIT_MAX_P:.16g}",
            f"{COMMON_AXIS_CURVE_MIN_P:.16g}",
            str(stats["num_points"]),
            str(stats["fit_degree"]),
            str(stats["failure_weight_sum"]),
            f"{coeff[0]:.16g}",
            f"{coeff[1]:.16g}",
            f"{coeff[2]:.16g}",
            f"{stats['rmse_log']:.16g}",
            f"{stats['max_abs_residual_log']:.16g}",
        ]
    )
    for record, y_value, yerr, fit_y, residual in zip(
        plot_rows, ys, yerrs, fit_at_x, residuals, strict=True
    ):
        p_value = float(record["p"])
        data_rows.append(
            [
                str(expected_d),
                str(block_size),
                code_name,
                parameter_label,
                f"{BB_FIT_MAX_P:.16g}",
                str(p_value <= BB_FIT_MAX_P),
                f"{p_value:.16g}",
                f"{y_value:.16g}",
                f"{yerr:.16g}",
                str(record["failures"]),
                str(record["shots"]),
                f"{fit_y:.16g}",
                f"{residual:.16g}",
            ]
        )
    print(
        f"BB fit {parameter_label}: p<={BB_FIT_MAX_P:g} d={expected_d} "
        f"points={stats['num_points']} degree={stats['fit_degree']} "
        f"c0={coeff[0]:.6g} c1={coeff[1]:.6g} c2={coeff[2]:.6g} "
        f"rmse_log={stats['rmse_log']:.4g}"
    )

if fitted_group_count == 0:
    raise RuntimeError("No BB code has enough points for fitting")
bb_coefficients_path = OUT_DIR / f"{BB_FIT_SAVE_PREFIX}_coefficients.csv"
bb_data_path = OUT_DIR / f"{BB_FIT_SAVE_PREFIX}_data.csv"
for path, rows_to_write in [
    (bb_coefficients_path, coefficient_rows),
    (bb_data_path, data_rows),
]:
    with path.open("w", newline="") as handle:
        csv.writer(handle).writerows(rows_to_write)
    print(f"wrote {path}")

## Surface-code X/Z average

In [ ]:
SURFACE_SUMMARY_TXT = REPO_ROOT / "surface_code" / "results" / "decode" / "summary.txt"


def load_surface_summary_txt(path: Path) -> list[dict[str, object]]:
    records: list[dict[str, object]] = []
    current_mode: str | None = None
    current_code: dict[str, object] = {}
    current: dict[str, object] | None = None
    section: str | None = None

    code_re = re.compile(
        r"^code=(?P<code>\S+)\s+(?P<label>\[\[[^\]]+\]\])\s+"
        r"distance=(?P<distance>\d+)\s+expected_d=(?P<d>\d+)\s*$"
    )
    condition_re = re.compile(r"^(?P<label>surface_\S+_[zx]basis_\S+)$")
    meta_re = re.compile(
        r"^meta:\s+basis=(?P<basis>\S+)\s+p=(?P<p>\S+)\s+cycles=(?P<cycles>\d+)\s+"
        r"detectors=(?P<detectors>\d+)\s+observables=(?P<observables>\d+)\s+"
        r"convergence_rate=(?P<convergence_rate>\S+)"
    )

    def finish_current() -> None:
        nonlocal current
        if current is not None:
            records.append(current)
            current = None

    for raw_line in path.read_text(encoding="utf-8").splitlines():
        stripped = raw_line.strip()
        if not stripped:
            continue
        if stripped.startswith("## mode="):
            current_mode = stripped.split("=", 1)[1].strip()
            continue

        code_match = code_re.match(stripped)
        if code_match:
            finish_current()
            current_code = {
                "code_name": code_match.group("code"),
                "parameter_label": code_match.group("label"),
                "distance": int(code_match.group("distance")),
                "expected_d": int(code_match.group("d")),
            }
            section = None
            continue

        condition_match = condition_re.match(stripped)
        if condition_match:
            finish_current()
            current = {
                "mode": current_mode,
                **current_code,
                "label": condition_match.group("label"),
                "basis": None,
                "p": None,
                "cycles": None,
                "detectors": None,
                "observables": None,
                "shots": None,
                "failures": None,
                "LER": None,
                "LER_per_cycle_per_logical": None,
            }
            section = None
            continue

        if current is None:
            continue

        meta_match = meta_re.match(stripped)
        if meta_match:
            current.update(
                {
                    "basis": meta_match.group("basis"),
                    "p": float(meta_match.group("p")),
                    "cycles": int(meta_match.group("cycles")),
                    "detectors": int(meta_match.group("detectors")),
                    "observables": int(meta_match.group("observables")),
                }
            )
            continue

        if stripped.endswith(":"):
            section = stripped[:-1]
            continue
        if section != "all shots":
            continue

        field_match = re.match(r"^(?P<key>\S+)\s+(?P<value>\S+)\s*$", stripped)
        if not field_match:
            continue
        key = field_match.group("key")
        value = field_match.group("value")
        if key in {"shots", "failures"}:
            current[key] = _as_int(value)
        elif key in {"LER", FIT_Y_COLUMN}:
            current[key] = _as_float(value)

    finish_current()
    return records


surface_source_records = load_surface_summary_txt(SURFACE_SUMMARY_TXT)
surface_average_records, surface_xz_average_incomplete = build_xz_average_records(
    surface_source_records, excluded_p_values=set()
)
print(
    f"Surface X/Z-basis average matched_points={len(surface_average_records)} "
    f"incomplete_points={len(surface_xz_average_incomplete)}"
)
for key in surface_xz_average_incomplete:
    print(f"Surface X/Z-basis average skipped incomplete key={key!r}")

SURFACE_AVERAGE_POINTS_PATH = OUT_DIR / "surface_xz_average_rates.csv"
if surface_average_records:
    write_xz_average_points(
        surface_average_records,
        SURFACE_AVERAGE_POINTS_PATH,
    )
    print(f"wrote {SURFACE_AVERAGE_POINTS_PATH}")

## Final two-panel figure

The shared plotter uses the exported tables and writes one PDF. Edit the settings below to change its appearance.

In [ ]:
# Edit these values to control the final two-panel figure.
# Panel sizes are [top panel, bottom panel].
height = [6, 5.4]
width = [7, 7]

# Font sizes are also [top panel, bottom panel].
axis_label_size = [14.5, 14.5]
tick_label_size = [12, 12]
legend_font_size = [11, 11]
# Independent font sizes for the external panel labels a and b.
subplot_a_label_font_size = 15
subplot_b_label_font_size = 15
subplot_label_size = [subplot_a_label_font_size, subplot_b_label_font_size]
# Horizontal positions in axes coordinates: [panel a, panel b].
subplot_label_x = [-0.125, -0.125]
subplot_label_y = [0.98, 0.98]

# Grid styling for [panel a, panel b].
grid_line_width = [0.5, 0.5]
grid_alpha = [0.3, 0.3]

# Vertical-axis limits as [(a_min, a_max), (b_min, b_max)].
y_limits = [(1e-15, 1e-2), (1e-15, 1e-4)]

# Axes layout margins as fractions of the full figure canvas.
figure_margins = {
    "left": 0.14,
    "right": 0.98,
    "bottom": 0.07,
    "top": 0.99,
}

# Extra exported-PDF margins in inches; each side is independent.
pdf_margin_inches = {
    "left": 0.01,
    "right": 0.05,
    "bottom": 0.01,
    "top": 0.01,
}

# All codes in one family use the same marker. Examples: 'o', 's', '^', 'v',
# 'd', 'P', 'X', '*'.
marker_choice = {
    "Cornucopia": "d",
    "BB": "*",
    "Surface": "s",
}

# Fitted-line styles by family: '-' solid, '--' dashed, ':' dotted,
# '-.' dash-dot.
line_style = {
    "Cornucopia": "-",
    "BB": "--",
    "Surface": "-.",
}

marker_size = {
    "Cornucopia": 10,
    "BB": 12,
    "Surface": 8,
}
line_width = {
    "Cornucopia": 1.4,
    "BB": 1.2,
    "Surface": 1.2,
}

# Style of the y=x guide in panel a.
break_even_style = {
    "color": "0.35",
    "linestyle": "--",
    "linewidth": 1.4,
}
break_even_x_limits = (1e-3, 4e-3)
panel_gap = 0.18  # leaves room for panel a's independent x-axis label

In [ ]:
from figures.logical_error_rates.plot_error_rates import plot_error_rates

display_settings = {
    "height": height,
    "width": width,
    "axis_label_size": axis_label_size,
    "tick_label_size": tick_label_size,
    "legend_font_size": legend_font_size,
    "subplot_label_size": subplot_label_size,
    "subplot_label_x": subplot_label_x,
    "subplot_label_y": subplot_label_y,
    "grid_line_width": grid_line_width,
    "grid_alpha": grid_alpha,
    "y_limits": y_limits,
    "figure_margins": figure_margins,
    "pdf_margin_inches": pdf_margin_inches,
    "marker_choice": marker_choice,
    "line_style": line_style,
    "marker_size": marker_size,
    "line_width": line_width,
    "break_even_style": break_even_style,
    "break_even_x_limits": break_even_x_limits,
    "panel_gap": panel_gap,
}
figure_path = plot_error_rates(OUT_DIR, OUT_DIR, display_settings=display_settings)
print(figure_path.relative_to(REPO_ROOT))